In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.product_product (
    product_id                          int,
    product_tmpl_id                     int,
    product_tmpl_name                   STRING,
    default_code                        STRING,
    barcode                             STRING,
    combination_indices                 STRING,
    standard_price                      DOUBLE,
    volume                              DOUBLE,
    weight                              DOUBLE,
    active                              BOOLEAN,
    can_image_variant_1024_be_zoomed    BOOLEAN,
    is_favorite                         BOOLEAN,
    is_in_selected_section_of_order     BOOLEAN,
    created_at                          TIMESTAMP,
    updated_at                          TIMESTAMP,
    dwh_loaded_at                       TIMESTAMP,
    dwh_source_db                       STRING
)
USING DELTA;

MERGE INTO silver.product_product AS target
USING (
    SELECT
        cast( pp.id As int)                                           AS product_id,
        cast( GET_JSON_OBJECT(pp.product_tmpl_id, '$[0]') As int)    AS product_tmpl_id,
        GET_JSON_OBJECT(pp.product_tmpl_id, '$[1]')    AS product_tmpl_name,
        pp.default_code                                 AS default_code,
        pp.barcode                                      AS barcode,
        pp.combination_indices                          AS combination_indices,
        CAST(pp.standard_price AS DOUBLE)               AS standard_price,
        CAST(pp.volume AS DOUBLE)                       AS volume,
        CAST(pp.weight AS DOUBLE)                       AS weight,
        CAST(pp.active AS BOOLEAN)                      AS active,
        CAST(pp.can_image_variant_1024_be_zoomed AS BOOLEAN) AS can_image_variant_1024_be_zoomed,
        CAST(pp.is_favorite AS BOOLEAN)                 AS is_favorite,
        CAST(pp.is_in_selected_section_of_order AS BOOLEAN) AS is_in_selected_section_of_order,
        CAST(pp.create_date AS TIMESTAMP)               AS created_at,
        CAST(pp.write_date  AS TIMESTAMP)               AS updated_at,
        current_timestamp()                             AS dwh_loaded_at,
        pp.dwh_source_db                                AS dwh_source_db
    FROM bronze.product_product pp
    WHERE CAST(pp.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'product_product'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN UPDATE SET
    target.product_tmpl_id                  = source.product_tmpl_id,
    target.product_tmpl_name                = source.product_tmpl_name,
    target.default_code                     = source.default_code,
    target.barcode                          = source.barcode,
    target.combination_indices              = source.combination_indices,
    target.standard_price                   = source.standard_price,
    target.volume                           = source.volume,
    target.weight                           = source.weight,
    target.active                           = source.active,
    target.can_image_variant_1024_be_zoomed = source.can_image_variant_1024_be_zoomed,
    target.is_favorite                      = source.is_favorite,
    target.is_in_selected_section_of_order  = source.is_in_selected_section_of_order,
    target.created_at                       = source.created_at,
    target.updated_at                       = source.updated_at,
    target.dwh_loaded_at                    = current_timestamp(),
    target.dwh_source_db                    = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.product_template (
    product_tmpl_id                     int,
    categ_id                            int,
    categ_name                          STRING,
    uom_id                              int,
    uom_name                            STRING,
    company_id                          int,
    company_name                        STRING,
    color                               INTEGER,
    type                                STRING,
    service_tracking                    STRING,
    default_code                        STRING,
    name                                STRING,
    description                         STRING,
    description_purchase                STRING,
    description_sale                    STRING,
    product_properties                  STRING,
    list_price                          DOUBLE,
    volume                              DOUBLE,
    weight                              DOUBLE,
    sale_ok                             BOOLEAN,
    purchase_ok                         BOOLEAN,
    active                              BOOLEAN,
    can_image_1024_be_zoomed            BOOLEAN,
    has_configurable_attributes         BOOLEAN,
    is_favorite                         BOOLEAN,
    property_account_income_id          int,
    property_account_income_name        STRING,
    property_account_expense_id         int,
    property_account_expense_name       STRING,
    service_type                        STRING,
    expense_policy                      STRING,
    invoice_policy                      STRING,
    sale_line_warn_msg                  STRING,
    created_at                          TIMESTAMP,
    updated_at                          TIMESTAMP,
    dwh_loaded_at                       TIMESTAMP,
    dwh_source_db                       STRING
)
USING DELTA;

MERGE INTO silver.product_template AS target
USING (
    SELECT
        cast( pt.id As int)                                 AS product_tmpl_id,

        cast( GET_JSON_OBJECT(pt.categ_id, '$[0]')  As int) AS categ_id,
        GET_JSON_OBJECT(pt.categ_id, '$[1]')                AS categ_name,

        cast( GET_JSON_OBJECT(pt.uom_id, '$[0]')  As int)   AS uom_id,
        GET_JSON_OBJECT(pt.uom_id, '$[1]')                  AS uom_name,

        cast( GET_JSON_OBJECT(pt.company_id, '$[0]') As int) AS company_id,
        GET_JSON_OBJECT(pt.company_id, '$[1]')               AS company_name,

        CAST(pt.color AS INTEGER)                           AS color,
        pt.type                                             AS type,
        pt.service_tracking                                 AS service_tracking,
        pt.default_code                                     AS default_code,
        pt.name                                             AS name,
        pt.description                                      AS description,
        pt.description_purchase                             AS description_purchase,
        pt.description_sale                                 AS description_sale,
        pt.product_properties                               AS product_properties,
        CAST(pt.list_price AS DOUBLE)                       AS list_price,
        CAST(pt.volume AS DOUBLE)                           AS volume,
        CAST(pt.weight AS DOUBLE)                           AS weight,
        CAST(pt.sale_ok AS BOOLEAN)                         AS sale_ok,
        CAST(pt.purchase_ok AS BOOLEAN)                     AS purchase_ok,
        CAST(pt.active AS BOOLEAN)                          AS active,
        CAST(pt.can_image_1024_be_zoomed AS BOOLEAN)        AS can_image_1024_be_zoomed,
        CAST(pt.has_configurable_attributes AS BOOLEAN)     AS has_configurable_attributes,
        CAST(pt.is_favorite AS BOOLEAN)                     AS is_favorite,
        cast( GET_JSON_OBJECT(pt.property_account_income_id, '$[0]') As int)  AS property_account_income_id,
        GET_JSON_OBJECT(pt.property_account_income_id, '$[1]')  AS property_account_income_name,
        cast( GET_JSON_OBJECT(pt.property_account_expense_id, '$[0]') As int) AS property_account_expense_id,
        GET_JSON_OBJECT(pt.property_account_expense_id, '$[1]') AS property_account_expense_name,
        pt.service_type                                     AS service_type,
        pt.expense_policy                                   AS expense_policy,
        pt.invoice_policy                                   AS invoice_policy,
        pt.sale_line_warn_msg                               AS sale_line_warn_msg,
        CAST(pt.create_date AS TIMESTAMP)                   AS created_at,
        CAST(pt.write_date  AS TIMESTAMP)                   AS updated_at,
        current_timestamp()                                 AS dwh_loaded_at,
        pt.dwh_source_db                                    AS dwh_source_db
    FROM bronze.product_template pt
    WHERE CAST(pt.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'product_template'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.product_tmpl_id = source.product_tmpl_id

WHEN MATCHED THEN UPDATE SET
    target.categ_id                     = source.categ_id,
    target.categ_name                   = source.categ_name,
    target.uom_id                       = source.uom_id,
    target.uom_name                     = source.uom_name,
    target.company_id                   = source.company_id,
    target.company_name                 = source.company_name,
    target.color                        = source.color,
    target.type                         = source.type,
    target.service_tracking             = source.service_tracking,
    target.default_code                 = source.default_code,
    target.name                         = source.name,
    target.description                  = source.description,
    target.description_purchase         = source.description_purchase,
    target.description_sale             = source.description_sale,
    target.product_properties           = source.product_properties,
    target.list_price                   = source.list_price,
    target.volume                       = source.volume,
    target.weight                       = source.weight,
    target.sale_ok                      = source.sale_ok,
    target.purchase_ok                  = source.purchase_ok,
    target.active                       = source.active,
    target.can_image_1024_be_zoomed     = source.can_image_1024_be_zoomed,
    target.has_configurable_attributes  = source.has_configurable_attributes,
    target.is_favorite                  = source.is_favorite,
    target.property_account_income_id   = source.property_account_income_id,
    target.property_account_income_name = source.property_account_income_name,
    target.property_account_expense_id  = source.property_account_expense_id,
    target.property_account_expense_name = source.property_account_expense_name,
    target.service_type                 = source.service_type,
    target.expense_policy               = source.expense_policy,
    target.invoice_policy               = source.invoice_policy,
    target.sale_line_warn_msg           = source.sale_line_warn_msg,
    target.created_at                   = source.created_at,
    target.updated_at                   = source.updated_at,
    target.dwh_loaded_at                = current_timestamp(),
    target.dwh_source_db                = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;

In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.product_category (
    categ_id                                int,
    parent_id                               int,
    parent_name                             STRING,
    name                                    STRING,
    complete_name                           STRING,
    parent_path                             STRING,
    product_properties_definition           STRING,
    property_account_income_categ_id        int,
    property_account_income_categ_name      STRING,
    property_account_expense_categ_id       int,
    property_account_expense_categ_name     STRING,
    created_at                              TIMESTAMP,
    updated_at                              TIMESTAMP,
    dwh_loaded_at                           TIMESTAMP,
    dwh_source_db                           STRING
)
USING DELTA;

MERGE INTO silver.product_category AS target
USING (
    SELECT
        cast( pc.id  As int)                                                          AS categ_id,
        cast( GET_JSON_OBJECT(pc.parent_id, '$[0]')    As int)                       AS parent_id,
        GET_JSON_OBJECT(pc.parent_id, '$[1]')                          AS parent_name,
        pc.name                                                         AS name,
        pc.complete_name                                                AS complete_name,
        pc.parent_path                                                  AS parent_path,
        pc.product_properties_definition                                AS product_properties_definition,
        cast( GET_JSON_OBJECT(pc.property_account_income_categ_id, '$[0]') As int)   AS property_account_income_categ_id,
        GET_JSON_OBJECT(pc.property_account_income_categ_id, '$[1]')   AS property_account_income_categ_name,
        cast( GET_JSON_OBJECT(pc.property_account_expense_categ_id, '$[0]') As int)  AS property_account_expense_categ_id,
        GET_JSON_OBJECT(pc.property_account_expense_categ_id, '$[1]')  AS property_account_expense_categ_name,
        CAST(pc.create_date AS TIMESTAMP)                               AS created_at,
        CAST(pc.write_date  AS TIMESTAMP)                               AS updated_at,
        current_timestamp()                                             AS dwh_loaded_at,
        pc.dwh_source_db                                                AS dwh_source_db
    FROM bronze.product_category pc
    WHERE CAST(pc.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'product_category'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.categ_id = source.categ_id

WHEN MATCHED THEN UPDATE SET
    target.parent_id                        = source.parent_id,
    target.parent_name                      = source.parent_name,
    target.name                             = source.name,
    target.complete_name                    = source.complete_name,
    target.parent_path                      = source.parent_path,
    target.product_properties_definition    = source.product_properties_definition,
    target.property_account_income_categ_id   = source.property_account_income_categ_id,
    target.property_account_income_categ_name = source.property_account_income_categ_name,
    target.property_account_expense_categ_id  = source.property_account_expense_categ_id,
    target.property_account_expense_categ_name = source.property_account_expense_categ_name,
    target.created_at                       = source.created_at,
    target.updated_at                       = source.updated_at,
    target.dwh_loaded_at                    = current_timestamp(),
    target.dwh_source_db                    = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.product_combo (
    combo_id                            int,
    company_id                          int,
    company_name                        STRING,
    name                                STRING,
    created_at                          TIMESTAMP,
    updated_at                          TIMESTAMP,
    dwh_loaded_at                       TIMESTAMP,
    dwh_source_db                       STRING
)
USING DELTA;

MERGE INTO silver.product_combo AS target
USING (
    SELECT
        cast( pc.id    As int)                                    AS combo_id,
        cast( GET_JSON_OBJECT(pc.company_id, '$[0]') As int)      AS company_id,
        GET_JSON_OBJECT(pc.company_id, '$[1]')      AS company_name,
        pc.name                                     AS name,
        CAST(pc.create_date AS TIMESTAMP)           AS created_at,
        CAST(pc.write_date  AS TIMESTAMP)           AS updated_at,
        current_timestamp()                         AS dwh_loaded_at,
        pc.dwh_source_db                            AS dwh_source_db
    FROM bronze.product_combo pc
    WHERE CAST(pc.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'product_combo'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.combo_id = source.combo_id

WHEN MATCHED THEN UPDATE SET
    target.company_id    = source.company_id,
    target.company_name  = source.company_name,
    target.name          = source.name,
    target.created_at    = source.created_at,
    target.updated_at    = source.updated_at,
    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


-- ============================================================

-- Table: silver.product_combo_item (5)
CREATE TABLE IF NOT EXISTS silver.product_combo_item (
    combo_item_id                       int,
    company_id                          int,
    company_name                        STRING,
    combo_id                            int,
    combo_name                          STRING,
    product_id                          int,
    product_name                        STRING,
    extra_price                         DOUBLE,
    created_at                          TIMESTAMP,
    updated_at                          TIMESTAMP,
    dwh_loaded_at                       TIMESTAMP,
    dwh_source_db                       STRING
)
USING DELTA;

MERGE INTO silver.product_combo_item AS target
USING (
    SELECT
        cast( pci.id As int)                                          AS combo_item_id,
        cast( GET_JSON_OBJECT(pci.company_id, '$[0]') As int)         AS company_id,
        GET_JSON_OBJECT(pci.company_id, '$[1]')                       AS company_name,
        cast( GET_JSON_OBJECT(pci.combo_id, '$[0]') As int)           AS combo_id,
        GET_JSON_OBJECT(pci.combo_id, '$[1]')                         AS combo_name,
        cast( GET_JSON_OBJECT(pci.product_id, '$[0]') As int)         AS product_id,
        GET_JSON_OBJECT(pci.product_id, '$[1]')         AS product_name,
        CAST(pci.extra_price AS DOUBLE)                 AS extra_price,
        CAST(pci.create_date AS TIMESTAMP)              AS created_at,
        CAST(pci.write_date  AS TIMESTAMP)              AS updated_at,
        current_timestamp()                             AS dwh_loaded_at,
        pci.dwh_source_db                               AS dwh_source_db
    FROM bronze.product_combo_item pci
    WHERE CAST(pci.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'product_combo_item'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.combo_item_id = source.combo_item_id

WHEN MATCHED THEN UPDATE SET
    target.company_id    = source.company_id,
    target.company_name  = source.company_name,
    target.combo_id      = source.combo_id,
    target.combo_name    = source.combo_name,
    target.product_id    = source.product_id,
    target.product_name  = source.product_name,
    target.extra_price   = source.extra_price,
    target.created_at    = source.created_at,
    target.updated_at    = source.updated_at,
    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;



In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.product_pricelist (
    pricelist_id                        int,
    currency_id                         int,
    currency_name                       STRING,
    company_id                          int,
    company_name                        STRING,
    name                                STRING,
    active                              BOOLEAN,
    created_at                          TIMESTAMP,
    updated_at                          TIMESTAMP,
    dwh_loaded_at                       TIMESTAMP,
    dwh_source_db                       STRING
)
USING DELTA;

MERGE INTO silver.product_pricelist AS target
USING (
    SELECT
        cast( pl.id As int)                                       AS pricelist_id,
        cast( GET_JSON_OBJECT(pl.currency_id, '$[0]') As int)     AS currency_id,
        GET_JSON_OBJECT(pl.currency_id, '$[1]')                   AS currency_name,
        cast( GET_JSON_OBJECT(pl.company_id, '$[0]') As int)      AS company_id,
        GET_JSON_OBJECT(pl.company_id, '$[1]')      AS company_name,
        pl.name                                     AS name,
        CAST(pl.active AS BOOLEAN)                  AS active,
        CAST(pl.create_date AS TIMESTAMP)           AS created_at,
        CAST(pl.write_date  AS TIMESTAMP)           AS updated_at,
        current_timestamp()                         AS dwh_loaded_at,
        pl.dwh_source_db                            AS dwh_source_db
    FROM bronze.product_pricelist pl
    WHERE CAST(pl.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'product_pricelist'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.pricelist_id = source.pricelist_id

WHEN MATCHED THEN UPDATE SET
    target.currency_id   = source.currency_id,
    target.currency_name = source.currency_name,
    target.company_id    = source.company_id,
    target.company_name  = source.company_name,
    target.name          = source.name,
    target.active        = source.active,
    target.created_at    = source.created_at,
    target.updated_at    = source.updated_at,
    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


-- ============================================================

-- Table: silver.product_pricelist_item (7)
CREATE TABLE IF NOT EXISTS silver.product_pricelist_item (
    pricelist_item_id                   int,
    pricelist_id                        int,
    pricelist_name                      STRING,
    company_id                          int,
    company_name                        STRING,
    currency_id                         int,
    currency_name                       STRING,
    categ_id                            int,
    categ_name                          STRING,
    product_tmpl_id                     int,
    product_tmpl_name                   STRING,
    product_id                          int,
    product_name                        STRING,
    base_pricelist_id                   int,
    base_pricelist_name                 STRING,
    applied_on                          STRING,
    display_applied_on                  STRING,
    base                                STRING,
    compute_price                       STRING,
    min_quantity                        DOUBLE,
    fixed_price                         DOUBLE,
    price_discount                      DOUBLE,
    price_round                         DOUBLE,
    price_surcharge                     DOUBLE,
    price_markup                        DOUBLE,
    price_min_margin                    DOUBLE,
    price_max_margin                    DOUBLE,
    percent_price                       DOUBLE,
    date_start                          TIMESTAMP,
    date_end                            TIMESTAMP,
    created_at                          TIMESTAMP,
    updated_at                          TIMESTAMP,
    dwh_loaded_at                       TIMESTAMP,
    dwh_source_db                       STRING
)
USING DELTA;

MERGE INTO silver.product_pricelist_item AS target
USING (
    SELECT
        cast( pli.id  As int)                                             AS pricelist_item_id,
        cast( GET_JSON_OBJECT(pli.pricelist_id, '$[0]') As int)           AS pricelist_id,
        GET_JSON_OBJECT(pli.pricelist_id, '$[1]')                         AS pricelist_name,
        cast( GET_JSON_OBJECT(pli.company_id, '$[0]')  As int)            AS company_id,
        GET_JSON_OBJECT(pli.company_id, '$[1]')                           AS company_name,
        cast( GET_JSON_OBJECT(pli.currency_id, '$[0]')  As int)           AS currency_id,
        GET_JSON_OBJECT(pli.currency_id, '$[1]')                          AS currency_name,
        cast( GET_JSON_OBJECT(pli.categ_id, '$[0]')  As int)              AS categ_id,
        GET_JSON_OBJECT(pli.categ_id, '$[1]')                             AS categ_name,
        cast( GET_JSON_OBJECT(pli.product_tmpl_id, '$[0]')  As int)       AS product_tmpl_id,
        GET_JSON_OBJECT(pli.product_tmpl_id, '$[1]')                      AS product_tmpl_name,
        cast( GET_JSON_OBJECT(pli.product_id, '$[0]')  As int)            AS product_id,
        GET_JSON_OBJECT(pli.product_id, '$[1]')                           AS product_name,
        cast( GET_JSON_OBJECT(pli.base_pricelist_id, '$[0]')   As int)    AS base_pricelist_id,
        GET_JSON_OBJECT(pli.base_pricelist_id, '$[1]')      AS base_pricelist_name,
        pli.applied_on                                      AS applied_on,
        pli.display_applied_on                              AS display_applied_on,
        pli.base                                            AS base,
        pli.compute_price                                   AS compute_price,
        CAST(pli.min_quantity AS DOUBLE)                    AS min_quantity,
        CAST(pli.fixed_price AS DOUBLE)                     AS fixed_price,
        CAST(pli.price_discount AS DOUBLE)                  AS price_discount,
        CAST(pli.price_round AS DOUBLE)                     AS price_round,
        CAST(pli.price_surcharge AS DOUBLE)                 AS price_surcharge,
        CAST(pli.price_markup AS DOUBLE)                    AS price_markup,
        CAST(pli.price_min_margin AS DOUBLE)                AS price_min_margin,
        CAST(pli.price_max_margin AS DOUBLE)                AS price_max_margin,
        CAST(pli.percent_price AS DOUBLE)                   AS percent_price,
        CAST(pli.date_start AS TIMESTAMP)                   AS date_start,
        CAST(pli.date_end AS TIMESTAMP)                     AS date_end,
        CAST(pli.create_date AS TIMESTAMP)                  AS created_at,
        CAST(pli.write_date  AS TIMESTAMP)                  AS updated_at,
        current_timestamp()                                 AS dwh_loaded_at,
        pli.dwh_source_db                                   AS dwh_source_db
    FROM bronze.product_pricelist_item pli
    WHERE CAST(pli.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'product_pricelist_item'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.pricelist_item_id = source.pricelist_item_id

WHEN MATCHED THEN UPDATE SET
    target.pricelist_id         = source.pricelist_id,
    target.pricelist_name       = source.pricelist_name,
    target.company_id           = source.company_id,
    target.company_name         = source.company_name,
    target.currency_id          = source.currency_id,
    target.currency_name        = source.currency_name,
    target.categ_id             = source.categ_id,
    target.categ_name           = source.categ_name,
    target.product_tmpl_id      = source.product_tmpl_id,
    target.product_tmpl_name    = source.product_tmpl_name,
    target.product_id           = source.product_id,
    target.product_name         = source.product_name,
    target.base_pricelist_id    = source.base_pricelist_id,
    target.base_pricelist_name  = source.base_pricelist_name,
    target.applied_on           = source.applied_on,
    target.display_applied_on   = source.display_applied_on,
    target.base                 = source.base,
    target.compute_price        = source.compute_price,
    target.min_quantity         = source.min_quantity,
    target.fixed_price          = source.fixed_price,
    target.price_discount       = source.price_discount,
    target.price_round          = source.price_round,
    target.price_surcharge      = source.price_surcharge,
    target.price_markup         = source.price_markup,
    target.price_min_margin     = source.price_min_margin,
    target.price_max_margin     = source.price_max_margin,
    target.percent_price        = source.percent_price,
    target.date_start           = source.date_start,
    target.date_end             = source.date_end,
    target.created_at           = source.created_at,
    target.updated_at           = source.updated_at,
    target.dwh_loaded_at        = current_timestamp(),
    target.dwh_source_db        = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;

In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.product_tag (
    tag_id                              int,
    color                               STRING,
    name                                STRING,
    visible_to_customers                BOOLEAN,
    created_at                          TIMESTAMP,
    updated_at                          TIMESTAMP,
    dwh_loaded_at                       TIMESTAMP,
    dwh_source_db                       STRING
)
USING DELTA;

MERGE INTO silver.product_tag AS target
USING (
    SELECT
        cast( pt.id  As int)                                   AS tag_id,
        pt.color                                AS color,
        pt.name                                 AS name,
        CAST(pt.visible_to_customers AS BOOLEAN) AS visible_to_customers,
        CAST(pt.create_date AS TIMESTAMP)       AS created_at,
        CAST(pt.write_date  AS TIMESTAMP)       AS updated_at,
        current_timestamp()                     AS dwh_loaded_at,
        pt.dwh_source_db                        AS dwh_source_db
    FROM bronze.product_tag pt
    WHERE CAST(pt.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'product_tag'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.tag_id = source.tag_id

WHEN MATCHED THEN UPDATE SET
    target.color                = source.color,
    target.name                 = source.name,
    target.visible_to_customers = source.visible_to_customers,
    target.created_at           = source.created_at,
    target.updated_at           = source.updated_at,
    target.dwh_loaded_at        = current_timestamp(),
    target.dwh_source_db        = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;
